# Lab 6 - a tree, a bag, and a forest

**Session 6.** Same split throughout. Watch variance fall as you average, and read
permutation importances rather than the impurity-based ones.

## 1. Data and a single tree

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# flip_y adds label noise - without it a deep tree has nothing to memorise WRONGLY,
# and the overfitting this lab is about never appears.
X, y = make_classification(n_samples=600, n_features=12, n_informative=5,
                           n_redundant=2, class_sep=0.9, flip_y=0.15,
                           random_state=2026)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y,
                                          random_state=2026)

full = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
print(f"unpruned tree: depth {full.get_depth()}, leaves {full.get_n_leaves()}")
print(f"  train accuracy {full.score(X_tr, y_tr):.3f}   test accuracy {full.score(X_te, y_te):.3f}")
print("Grown to purity: zero training error, and the gap IS the diagnosis.")

## 2. Depth is the flexibility dial

In [ ]:
print("depth  train   test")
for d in (1, 2, 3, 5, 8, None):
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    label = "none" if d is None else str(d)
    print(f"{label:>5}  {t.score(X_tr, y_tr):.3f}  {t.score(X_te, y_te):.3f}")

## 3. Instability: the same model, five resamples

This is the variance that bagging exists to remove.

In [ ]:
rng = np.random.default_rng(2026)
scores = []
for _ in range(5):
    boot = rng.integers(0, len(X_tr), len(X_tr))
    t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr[boot], y_tr[boot])
    scores.append(t.score(X_te, y_te))
    print(f"  root split on feature {t.tree_.feature[0]:2d} "
          f"at {t.tree_.threshold[0]:7.3f}   test accuracy {scores[-1]:.3f}")
print(f"\nspread across resamples: {max(scores) - min(scores):.3f}")

## 4. Bagging, and a random forest

In [ ]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

bag = BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=300,
                        oob_score=True, random_state=0).fit(X_tr, y_tr)
rf = RandomForestClassifier(n_estimators=300, max_features="sqrt", oob_score=True,
                            random_state=0).fit(X_tr, y_tr)

print(f"single tree (depth 3) test accuracy : "
      f"{DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr).score(X_te, y_te):.3f}")
print(f"bagging     test {bag.score(X_te, y_te):.3f}   OOB {bag.oob_score_:.3f}")
print(f"forest      test {rf.score(X_te, y_te):.3f}   OOB {rf.oob_score_:.3f}")
print("\nOOB is a free validation estimate: the ~37% of rows each tree never saw.")

## 5. Boosting, briefly

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

for lr in (0.5, 0.1, 0.03):
    gb = HistGradientBoostingClassifier(
        learning_rate=lr, max_iter=500, early_stopping=True,
        validation_fraction=0.15, random_state=0).fit(X_tr, y_tr)
    print(f"learning rate {lr:<5} stopped at {gb.n_iter_:3d} trees   "
          f"test accuracy {gb.score(X_te, y_te):.3f}")
print("\nLower rate, more trees - and early stopping decides how many.")

## 6. Importances: impurity versus permutation

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_te, y_te, n_repeats=20, random_state=0)
order = np.argsort(perm.importances_mean)[::-1]

print("feature  impurity  permutation (mean +/- sd)")
for j in order[:6]:
    print(f"  {j:5d}    {rf.feature_importances_[j]:.3f}     "
          f"{perm.importances_mean[j]:.3f} +/- {perm.importances_std[j]:.3f}")
print("\nThe rankings usually differ. Trust the permutation one: it measures the drop in")
print("SCORE when a column is made useless, not how often the fitting used it.")

## Exercises

1. **How many trees?** Score the forest at `n_estimators` in `[1, 5, 25, 100, 300]`. Where
   do the returns stop? Is more ever worse?
2. **`max_features`.** Refit with `max_features=None` (i.e. plain bagging of deep trees) and
   compare. Explain the difference in terms of correlation between trees.
3. **Overfit the booster.** Set `early_stopping=False, max_iter=2000, learning_rate=0.5`.
   What happens to test accuracy, and what does that tell you about how boosting differs from
   a forest?